In [3]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score,KFold,GridSearchCV,train_test_split
from sklearn.linear_model import LinearRegression ,Ridge,Lasso

In [4]:
df=pd.read_csv("gurgaon_properties_post_feature_selection_v2.csv").drop(columns=['store room','floor_category','balcony'])

In [5]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servent room,furnishing_type,luxury_category
0,flat,sector 92,0.21,1.0,1,Relatively New,336.0,0,0,Medium
1,flat,sector 59,5.90,4.0,4,Moderately old,5350.0,0,0,Medium
2,flat,sector 1,0.90,3.0,3,Moderately old,1900.0,0,0,Low
3,house,sector 15,10.00,5.0,5,Old Property,4518.0,0,0,Low
4,flat,sector 48,0.72,2.0,2,Relatively New,1165.0,0,0,High


In [6]:
# Numerical = bedRoom, bathroom, built_up_area, servant room
# Ordinal = property_type, furnishing_type, luxury_category 
# OHE = sector, agePossession

In [7]:
df['agePossession']=df['agePossession'].replace({
    'Relatively New':'new',
        'Moderately old':'old',
        'New Property' : 'new',
        'Old Property' : 'old',
        'Under Construction' : 'under construction'
                                                })

In [8]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servent room,furnishing_type,luxury_category
0,flat,sector 92,0.21,1.0,1,new,336.0,0,0,Medium
1,flat,sector 59,5.90,4.0,4,old,5350.0,0,0,Medium
2,flat,sector 1,0.90,3.0,3,old,1900.0,0,0,Low
3,house,sector 15,10.00,5.0,5,old,4518.0,0,0,Low
4,flat,sector 48,0.72,2.0,2,new,1165.0,0,0,High


In [9]:
df['property_type']=df['property_type'].replace({'flat':0,'house':1})

C:\Users\sarth\AppData\Local\Temp\ipykernel_24168\3053022205.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['property_type']=df['property_type'].replace({'flat':0,'house':1})


In [10]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servent room,furnishing_type,luxury_category
0,0,sector 92,0.21,1.0,1,new,336.0,0,0,Medium
1,0,sector 59,5.90,4.0,4,old,5350.0,0,0,Medium
2,0,sector 1,0.90,3.0,3,old,1900.0,0,0,Low
3,1,sector 15,10.00,5.0,5,old,4518.0,0,0,Low
4,0,sector 48,0.72,2.0,2,new,1165.0,0,0,High


In [11]:
df['luxury_category']=df['luxury_category'].replace({'Low':0,"Medium":1,"High":2})

C:\Users\sarth\AppData\Local\Temp\ipykernel_24168\1761035597.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['luxury_category']=df['luxury_category'].replace({'Low':0,"Medium":1,"High":2})


In [12]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servent room,furnishing_type,luxury_category
0,0,sector 92,0.21,1.0,1,new,336.0,0,0,1
1,0,sector 59,5.90,4.0,4,old,5350.0,0,0,1
2,0,sector 1,0.90,3.0,3,old,1900.0,0,0,0
3,1,sector 15,10.00,5.0,5,old,4518.0,0,0,0
4,0,sector 48,0.72,2.0,2,new,1165.0,0,0,2


In [13]:
new_df=pd.get_dummies(df,columns=['sector','agePossession'],drop_first=True)

In [14]:
X=new_df.drop(columns='price')
y=new_df['price']
y_log=np.log1p(y)
X.shape,y_log

((3639, 102),
 0       0.190620
 1       1.931521
 2       0.641854
 3       2.397895
 4       0.542324
           ...   
 3634    1.208960
 3635    1.131402
 3636    1.945910
 3637    0.717840
 3638    0.970779
 Name: price, Length: 3639, dtype: float64)

In [15]:
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

In [16]:
X_scaled=pd.DataFrame(X_scaled,columns=X.columns)

In [17]:
X_scaled

,property_type,bedRoom,bathroom,built_up_area,servent room,furnishing_type,luxury_category,sector_sector 10,sector_sector 102,sector_sector 103,...,sector_sector 89,sector_sector 9,sector_sector 90,sector_sector 91,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,agePossession_old,agePossession_under construction
0,-0.536788,-1.664363,-1.555934,-1.241375,-0.309442,-0.681065,0.460524,-0.05752,-0.175722,-0.108057,...,-0.127266,-0.088057,-0.157422,-0.068509,5.979764,-0.049793,-0.123879,-0.108057,-0.615150,-0.186249
1,-0.536788,0.722951,0.505740,2.767521,-0.309442,-0.681065,0.460524,-0.05752,-0.175722,-0.108057,...,-0.127266,-0.088057,-0.157422,-0.068509,-0.167231,-0.049793,-0.123879,-0.108057,1.625621,-0.186249
2,-0.536788,-0.072820,-0.181485,0.009106,-0.309442,-0.681065,-0.964516,-0.05752,-0.175722,-0.108057,...,-0.127266,-0.088057,-0.157422,-0.068509,-0.167231,-0.049793,-0.123879,-0.108057,1.625621,-0.186249
3,1.862932,1.518723,1.192964,2.102304,-0.309442,-0.681065,-0.964516,-0.05752,-0.175722,-0.108057,...,-0.127266,-0.088057,-0.157422,-0.068509,-0.167231,-0.049793,-0.123879,-0.108057,1.625621,-0.186249
4,-0.536788,-0.868591,-0.868709,-0.578556,-0.309442,-0.681065,1.885565,-0.05752,-0.175722,-0.108057,...,-0.127266,-0.088057,-0.157422,-0.068509,-0.167231,-0.049793,-0.123879,-0.108057,-0.615150,-0.186249
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3634,-0.536788,-0.072820,0.505740,0.549596,-0.309442,1.563860,0.460524,-0.05752,-0.175722,-0.108057,...,-0.127266,-0.088057,-0.157422,-0.068509,-0.167231,-0.049793,-0.123879,-0.108057,-0.615150,-0.186249
3635,-0.536788,-0.868591,-0.868709,-0.674501,-0.309442,-0.681065,-0.964516,-0.05752,-0.175722,-0.108057,...,-0.127266,-0.088057,-0.157422,-0.068509,-0.167231,-0.049793,-0.123879,-0.108057,-0.615150,-0.186249
3636,-0.536788,-0.072820,0.505740,0.355308,-0.309442,1.563860,-0.964516,-0.05752,-0.175722,-0.108057,...,-0.127266,-0.088057,-0.157422,-0.068509,-0.167231,-0.049793,-0.123879,-0.108057,1.625621,-0.186249
3637,-0.536788,-0.072820,-0.181485,-0.165993,-0.309442,1.563860,0.460524,-0.05752,-0.175722,-0.108057,...,-0.127266,-0.088057,-0.157422,-0.068509,-0.167231,-0.049793,-0.123879,-0.108057,-0.615150,-0.186249


In [18]:
model=LinearRegression()
k_fold=KFold(n_splits=10,random_state=42, shuffle=True)
score=cross_val_score(model,X_scaled,y_log,cv=k_fold,scoring='r2')

In [19]:
 score.mean(),score.std(),score

(np.float64(0.8346873415604342),
 np.float64(0.024487706466719644),
 array([0.83089491, 0.81117786, 0.82688001, 0.84628788, 0.85414357,
        0.82912905, 0.7783982 , 0.84487925, 0.86298845, 0.86209424]))

In [20]:
# apply lasso and ridge and then do hyperparameter tuning
lasso=Lasso()
lasso_params={'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
grid=GridSearchCV(lasso,lasso_params,cv=k_fold,scoring='r2')

In [21]:
X_train,X_test,y_train,y_test=train_test_split(X_scaled,y_log,test_size=0.2,random_state=42)

In [22]:
grid.fit(X_train,y_train)

,estimator,Lasso()
,param_grid,"{'alpha': [0.001, 0.01, ...]}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,0.001


In [23]:
grid.best_params_

{'alpha': 0.001}

In [24]:
grid.best_score_

np.float64(0.8340577226386177)

In [25]:
ridge=Ridge()
ridge_params={'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
grid=GridSearchCV(ridge,ridge_params,cv=k_fold,scoring='r2')

In [26]:
grid.fit(X_train,y_train)

,estimator,Ridge()
,param_grid,"{'alpha': [0.001, 0.01, ...]}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,1


In [27]:
grid.best_params_

{'alpha': 1}

In [28]:
grid.best_score_

np.float64(0.8353354494462344)

In [29]:
model=Ridge(alpha=1)
k_fold=KFold(n_splits=10,random_state=42, shuffle=True)
score=cross_val_score(model,X_scaled,y_log,cv=k_fold,scoring='r2')

In [30]:
score.mean(),score.std()

(np.float64(0.8346881421047467), np.float64(0.02445438482255275))

In [31]:
ridge.fit(X_scaled,y_log)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [32]:
ridge.coef_

array([ 0.07284487,  0.04927071,  0.08943429,  0.21852534,  0.07576352,
        0.01149813,  0.00965537,  0.00973699,  0.06627671,  0.02127553,
        0.03289995,  0.00024211,  0.0279733 ,  0.01346847,  0.04521508,
        0.05482202,  0.00080987,  0.02756281,  0.03449033,  0.04764122,
        0.05822391,  0.00313149, -0.00153374,  0.04265896,  0.00846232,
        0.02367941,  0.03015558, -0.00041963,  0.05251888,  0.0117796 ,
        0.03454188,  0.07279335,  0.08706783,  0.00889699,  0.06129972,
        0.00649643,  0.01202653,  0.03209931,  0.05620789,  0.02004106,
        0.03218003,  0.01833067,  0.0152738 ,  0.00823614,  0.0143345 ,
        0.02415878,  0.0806342 ,  0.03276372,  0.02628055,  0.02464556,
        0.04719074,  0.05214322,  0.00320408,  0.0871518 ,  0.00484964,
        0.01469603,  0.0615159 ,  0.06136377,  0.0211113 ,  0.0315836 ,
        0.03649403,  0.03313713,  0.04364734, -0.00239161,  0.03802002,
        0.06134768,  0.05724681,  0.06136399,  0.10811256,  0.07

In [33]:
ridge=Ridge(alpha=0.01)
ridge.fit(X_scaled,y_log)

,alpha,0.01
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [34]:
X.shape

(3639, 102)

In [35]:
coef_df=pd.DataFrame(ridge.coef_.reshape(1,102),columns=X.columns).stack().reset_index().drop(columns=['level_0']).rename (columns={'level_1':'feature',0:'coef'})

In [36]:
coef_df

,feature,coef
0,property_type,0.072787
1,bedRoom,0.049238
2,bathroom,0.089436
3,built_up_area,0.218648
4,servent room,0.075755
...,...,...
97,sector_sector 93,0.007062
98,sector_sector 95,0.002059
99,sector_sector 99,0.014796
100,agePossession_old,0.017830


In [88]:
import statsmodels.api as sm 
x_with_const =sm.add_constant(X_scaled)
model=sm.OLS(y_log,x_with_const).fit(-)
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.846
Model:                            OLS   Adj. R-squared:                  0.842
Method:                 Least Squares   F-statistic:                     191.0
Date:                Sat, 21 Mar 2026   Prob (F-statistic):               0.00
Time:                        18:54:21   Log-Likelihood:                 360.56
No. Observations:                3639   AIC:                            -515.1
Df Residuals:                    3536   BIC:                             123.4
Df Model:                         102                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
const   

In [89]:
y_log.std()

0.5592449060829198

In [90]:
X_scaled['bedRoom'].std()

1.0001374287095028

In [91]:
b=0.049238*(0.5592449060829198/1.0001374287095028)

In [93]:
b=np.expm1(b)

In [94]:
b
# this means on incresing the no of bedroom our price incresed by 0.27 lakh

np.float64(0.02791483364783144)

In [95]:
# do it for built up  area  
X_scaled['built_up_area'].std()

1.000137428709508

In [96]:
0.049238*(0.5592449060829198/1.000137428709508)

0.027532316954921924